<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/12-case-studies-and-capstone/04-deepfake-image-detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study D — Deepfake detection for EU AI Act Art. 50(4): detector, VLM, or both?

**Goal:** Compare four ways to flag **undisclosed deepfakes**, the thing Article 50(4) of the EU AI Act actually regulates, on a handful of hand-picked images. Then point the same comparison at any image you like.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

> **The thesis, up front:** "is this image AI-generated?" and "does this image break Art. 50(4)?" are different questions. The first is perception, where a trained detector should beat a general model. The second also needs *meaning* (does it depict a real person or event? is it evidently satire?) and *provenance* (was it already disclosed?), which no pixel classifier answers. So the likely winner is a hybrid.

The notebook runs in six steps: **1** the ask, **2** the data, **3** the methods, **4** each tool on the data, **5** all four systems compared, **6** your own image.

Multimodal stays off the core spine ([curriculum decision #3](../docs/curriculum-plan.md)); this case study composes the evals ideas ([02](../02-evals-basics/01-measuring-outputs.ipynb), [04](../04-evals/01-golden-sets.ipynb)) with one new modality. The detectors are ordinary image classifiers, so they load with `transformers`, the tool you'd actually reach for.

## Setup

Get a free key at [console.groq.com](https://console.groq.com/); in Colab add it via the **key icon** → secret `GROQ_API_KEY`. (Full walkthrough: [00-setup](../00-setup/00-environment.ipynb).) An `HF_TOKEN` secret (a free Hugging Face read token) is optional; it raises Hugging Face's rate limits.

Runs on a CPU runtime. A full run downloads two detectors (~0.7 GB) and 7 example images, and makes about 10 VLM calls (~1,600 tokens each), well inside Groq's free daily limit.

In [ ]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git@v0.1.0"

In [ ]:
import base64, hashlib, io, json, os, re, time, urllib.error, urllib.parse, urllib.request
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont
from PIL.PngImagePlugin import PngInfo
from aien import setup

client, TEXT_MODEL = setup()
VISION_MODEL = "qwen/qwen3.8-27b"  # Groq's image-capable model; if calls 404, see console.groq.com/docs/vision
MAX_SIDE = 512                     # every model sees the image downscaled to this
CACHE_PATH = "case_d_cache.json"   # VLM answers, so a rerun never repeats a paid call
SAMPLE_DIR = "case_d_samples"      # the downloaded example images; see the license note in step 2

def http_get(url, headers=None, tries=5):
    """GET with backoff. Hugging Face throttles bursts: on 429 wait as long as the server asks."""
    headers = {"User-Agent": "ai-engineer-notebooks (case study D)", **(headers or {})}
    for attempt in range(1, tries + 1):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=60) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code not in (429, 500, 502, 503, 504) or attempt == tries:
                raise
            ra = e.headers.get("Retry-After") or ""
            wait = int(ra) if ra.isdigit() else 10 * attempt
            print(f"   HTTP {e.code}; waiting {wait}s ({attempt}/{tries})")
        except (urllib.error.URLError, TimeoutError) as e:
            if attempt == tries:
                raise
            wait = 10 * attempt
            print(f"   {type(e).__name__}; waiting {wait}s ({attempt}/{tries})")
        time.sleep(wait)

def view(raw, side=MAX_SIDE):
    """The pixels a model sees: decoded, RGB, downscaled. File metadata doesn't survive this."""
    img = Image.open(io.BytesIO(raw))
    img.draft("RGB", (side, side))     # JPEGs decode straight at reduced size
    img = img.convert("RGB")
    img.thumbnail((side, side))
    return img

def font(size):
    try:
        return ImageFont.load_default(size=size)
    except TypeError:                  # Pillow < 10.1
        return ImageFont.load_default()

CACHE = json.load(open(CACHE_PATH)) if os.path.exists(CACHE_PATH) else {}

def cached(key, fn):
    if key not in CACHE:
        CACHE[key] = fn()
        with open(CACHE_PATH, "w") as f:
            json.dump(CACHE, f)
    return CACHE[key]

## 1. The ask

A trust-and-safety team: *"Article 50 applies from 2 August 2026. Find the deepfakes on our platform that should have been labelled and weren't."* Before building anything, read the article ([11/01 scoping](../11-customer-craft/01-scoping-and-discovery.ipynb)).

Article 50(4) puts the duty on **deployers** (whoever publishes the content): disclose that a deepfake is generated or manipulated. Evidently artistic, satirical or fictional works only need a limited disclosure. A **deepfake** (Art. 3(60)) is generated or manipulated content that resembles existing persons, objects, places, entities or events and would falsely appear authentic. So an image is a 50(4) problem only when **all three** hold:

1. it is AI-generated or AI-manipulated (*perception*);
2. it depicts a real, identifiable entity or event, and is not evidently fiction or satire (*meaning*);
3. the published file carries **no disclosure**: no visible label, no machine-readable marking (*provenance*).

Separately, Art. 50(2) obliges the **providers** of generators to mark their outputs in a machine-readable way, and the Commission's June 2026 Code of Practice asks for at least two layered marks (e.g. metadata plus watermark). That's why condition 3 is partly checkable from the file itself.

**Scope:** still images only (50(4) also covers audio and video). The system **flags for human review**; deciding that something is a violation is a legal call, and nothing here is legal advice.

## 2. The data

**[OpenFake](https://huggingface.co/datasets/ComplexDataLab/OpenFake)** (`core` subset) has 2.4M real and AI-generated images from 75 sources, each labelled `real` or `fake`, with the generator's name. Real-entity deepfakes are rare in it (about 1 in 40 fakes), so instead of sampling we **pick** seven images from its test split:

- two realistic fakes of real entities: a Real Madrid press conference and *The Daily Show* studio;
- an AI image of an anonymous man in a "Bernie for President" shirt (a judgement call, see step 5);
- an anime illustration (evident fiction);
- three real photos.

Each image is fetched by its row number through Hugging Face's dataset viewer API: two small requests per image, saved to disk, so a rerun downloads nothing.

> **⚠️ Production reality —** OpenFake is **CC BY-NC 4.0** (non-commercial) and contains images of real public figures. Keep `SAMPLE_DIR` private, don't redistribute it, and delete it when you're done.

In [ ]:
DATASET_ID, SUBSET, SPLIT = "ComplexDataLab/OpenFake", "core", "test"
VIEWER = "https://datasets-server.huggingface.co"   # Hugging Face's dataset viewer API
HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:                  # not in Colab, or no HF_TOKEN secret: anonymous is fine
    pass
HF_HEADERS = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}

EXAMPLES = [   # row in OpenFake core/test, and what it shows
    {"row": 6831, "what": "Real Madrid press conference"},
    {"row": 29127, "what": "The Daily Show studio"},
    {"row": 3251, "what": "anonymous man, Bernie shirt"},
    {"row": 6814, "what": "anime girl on a beach"},
    {"row": 6811, "what": "bearded dragon"},
    {"row": 6851, "what": "Mini Cooper under a tarp"},
    {"row": 6804, "what": "parking-garage mirror"},
]

def fetch_example(ex):
    """Image bytes plus the dataset's label and generator, from disk if we already have them."""
    path = os.path.join(SAMPLE_DIR, f"{SPLIT}-{ex['row']}")
    if not os.path.exists(path + ".json"):
        q = urllib.parse.urlencode({"dataset": DATASET_ID, "config": SUBSET, "split": SPLIT,
                                    "offset": ex["row"], "length": 1})
        row = json.loads(http_get(f"{VIEWER}/rows?{q}", HF_HEADERS))["rows"][0]["row"]
        os.makedirs(SAMPLE_DIR, exist_ok=True)
        with open(path + ".img", "wb") as f:
            f.write(http_get(row["image"]["src"]))   # a signed link to this one image
        with open(path + ".json", "w") as f:
            json.dump({"label": str(row["label"]).lower(), "generator": row.get("model")}, f)
    ex.update(json.load(open(path + ".json")))
    ex["raw"] = open(path + ".img", "rb").read()

for ex in EXAMPLES:
    fetch_example(ex)

# The seven images, captioned with the dataset's own label.
W = 200
sheet = Image.new("RGB", (W * len(EXAMPLES), W + 40), "white")
draw = ImageDraw.Draw(sheet)
for i, ex in enumerate(EXAMPLES):
    sheet.paste(view(ex["raw"], W - 6), (W * i + 3, 3))
    draw.text((W * i + 4, W + 2), f"{ex['label'].upper()}  row {ex['row']}", fill="red" if ex["label"] == "fake" else "green", font=font(15))
    draw.text((W * i + 4, W + 20), str(ex["generator"])[:26], fill="black", font=font(13))
display(sheet)
for ex in EXAMPLES:
    print(f"row {ex['row']:>6}  {ex['label']:4}  {str(ex['generator']):22}  {ex['what']}")

### From images to cases

Condition 3 is about the **published file**, so we publish each image ourselves. Most go out with no disclosure. The two real-entity fakes also go out a second time, disclosed: one with a visible "AI-generated" label drawn on the image, one with an AI mark written into its metadata. The same picture is then a violation in one file and not in the other.

That gives nine cases. For each one we write down the **gold** answer, what a careful human reviewer would decide: violation = fake **and** real entity **and** not fiction **and** not disclosed. Conditions 2 and 3 are our labels, by hand, which is what a small golden set should be ([04/01](../04-evals/01-golden-sets.ipynb)).

In [ ]:
XMP = ('<x:xmpmeta xmlns:x="adobe:ns:meta/"><rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#">'
       '<rdf:Description xmlns:Iptc4xmpExt="http://iptc.org/std/Iptc4xmpExt/2008-02-29/" '
       'Iptc4xmpExt:DigitalSourceType="http://cv.iptc.org/newscodes/digitalsourcetype/trainedAlgorithmicMedia"/>'
       '</rdf:RDF></x:xmpmeta>')

def png_bytes(img, pnginfo=None):
    buf = io.BytesIO()
    img.save(buf, format="PNG", pnginfo=pnginfo)
    return buf.getvalue()

def with_metadata(raw):
    """Re-save as PNG with the IPTC 'made by AI' term in its XMP metadata."""
    info = PngInfo()
    info.add_itxt("XML:com.adobe.xmp", XMP)
    return png_bytes(view(raw, 1024), info)

def with_visible_label(raw, text="AI-generated"):
    """Draw a black 'AI-generated' box in the top-left corner."""
    img = view(raw, 1024)
    draw, size = ImageDraw.Draw(img), max(14, img.height // 18)
    x0, y0, x1, y1 = draw.textbbox((0, 0), text, font=font(size))
    pad = size // 3
    draw.rectangle([8, 8, 8 + x1 - x0 + 2 * pad, 8 + y1 - y0 + 2 * pad], fill="black")
    draw.text((8 + pad - x0, 8 + pad - y0), text, fill="white", font=font(size))
    return png_bytes(img)

PUBLISH = {"none": lambda raw: raw, "visible label": with_visible_label, "metadata": with_metadata}
by_row = {ex["row"]: ex for ex in EXAMPLES}
CASES = [   # row, how it's published, and our hand labels for condition 2
    {"row": 6831,  "disclosure": "none",          "real_entity": True,  "fiction": False},
    {"row": 6831,  "disclosure": "visible label", "real_entity": True,  "fiction": False},
    {"row": 29127, "disclosure": "none",          "real_entity": True,  "fiction": False},
    {"row": 29127, "disclosure": "metadata",      "real_entity": True,  "fiction": False},
    {"row": 3251,  "disclosure": "none",          "real_entity": False, "fiction": False},
    {"row": 6814,  "disclosure": "none",          "real_entity": False, "fiction": True},
    {"row": 6811,  "disclosure": "none",          "real_entity": False, "fiction": False},
    {"row": 6851,  "disclosure": "none",          "real_entity": False, "fiction": False},
    {"row": 6804,  "disclosure": "none",          "real_entity": False, "fiction": False},
]
for c in CASES:
    ex = by_row[c["row"]]
    c["name"] = f"{ex['what']} [{c['disclosure']}]"
    c["label"] = ex["label"]
    c["published"] = PUBLISH[c["disclosure"]](ex["raw"])
    c["gold"] = c["label"] == "fake" and c["real_entity"] and not c["fiction"] and c["disclosure"] == "none"

print(f"{'case':46} {'label':5} {'real entity':>11} {'fiction':>7}  gold")
for c in CASES:
    print(f"{c['name']:46} {c['label']:5} {str(c['real_entity']):>11} {str(c['fiction']):>7}  {'VIOLATION' if c['gold'] else '-'}")
display(view(CASES[1]["published"], 240))   # what the visible-label copy looks like

## 3. The methods

Three tools, each answering the condition it's suited to:

| Tool | What it is | Answers | Cost |
|---|---|---|---|
| **A — detector** | an image classifier trained to tell real photos from AI images; outputs `p_fake` | condition 1: is it AI-generated? | ~0.2 s on a CPU, free |
| **B — VLM** | a vision-language model asked to look at the image and fill in a JSON form | conditions 1 and 2, plus a *visible* AI label | one API call per image |
| **D — provenance** | read the file's bytes for a machine-readable "made by AI" mark | condition 3: was it disclosed in the file? | microseconds |

**D in one sentence:** it doesn't look at the picture at all, only at the file's metadata. Generators like ChatGPT and Gemini stamp their output files with the IPTC term `trainedAlgorithmicMedia` ("made by a trained model") and a signed C2PA manifest (a "content credential"). If the mark is there, the file has disclosed itself as AI, so it can't be an *undisclosed* deepfake. D is the cheapest tool, and also the most fragile: a screenshot or re-save throws the metadata away.

From the three tools we build four systems, each a different answer to "flag this for review?":

| System | Flags when |
|---|---|
| **A only** | the detector says fake (the naive build: every fake is a "violation") |
| **A + D** | detector says fake **and** no metadata mark |
| **B end to end** | the VLM says fake, real entity, not fiction, no visible label (one model does everything it can see) |
| **C hybrid** | detector says fake **and** VLM says real entity, not fiction, no visible label **and** no metadata mark |

B and C use the same VLM answer; they differ only in *who decides "is it fake"*: the VLM itself, or the detector.

## 4. The tools, one at a time

Each tool is loaded and run on the nine cases straight away, so you see what it says before anything is combined.

### D — provenance check

We look for the IPTC term and for an embedded C2PA box. Finding a C2PA box is not *verifying* it: that needs the signature checked against a trust list (exercise 5). Invisible watermarks like SynthID can't be checked here at all.

In [ ]:
def provenance(raw):
    return {"ai_metadata": b"trainedAlgorithmicMedia" in raw,
            "c2pa_manifest": b"jumb" in raw and b"c2pa" in raw}

for c in CASES:
    c["prov"] = provenance(c["published"])
    print(f"{c['name']:46} {c['prov']}")

Only the copy we published with metadata carries a mark. The OpenFake originals have none: the dataset stores plain images, the way most re-uploaded content looks on a platform.

### A — off-the-shelf detectors

You don't have to train a detector to use one. Two pretrained classifiers, both used as-is with a 0.5 threshold:

- **`A_openfake`**: the SwinV2 detector the [OpenFake](https://huggingface.co/datasets/ComplexDataLab/OpenFake) authors ship with their dataset, at [`bendavidsteel/openfake-swinv2-base`](https://huggingface.co/bendavidsteel/openfake-swinv2-base). Trained on OpenFake's train split, so it knows these generators; expect it to look better here than it would on your platform.
- **`A_generic`**: [`umm-maybe/AI-image-detector`](https://huggingface.co/umm-maybe/AI-image-detector), one of the most-downloaded AI image detectors on the Hub, trained in 2022 on older generators. What you get by grabbing the popular model.

In [ ]:
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
THRESHOLD = 0.5

class Detector:
    def __init__(self, repo, fake_label, temperature=1.0):
        self.proc = AutoImageProcessor.from_pretrained(repo)
        self.model = AutoModelForImageClassification.from_pretrained(repo).to(DEVICE).eval()
        self.fake = {v.lower(): int(k) for k, v in self.model.config.id2label.items()}[fake_label]
        self.temperature = temperature

    @torch.no_grad()
    def __call__(self, raws):
        x = self.proc(images=[view(r) for r in raws], return_tensors="pt").to(DEVICE)
        p = (self.model(**x).logits.float() / self.temperature).softmax(-1)[:, self.fake]
        return p.cpu().tolist()

DETECTORS = {
    "A_openfake": Detector("bendavidsteel/openfake-swinv2-base", "fake", temperature=2.0),  # its card: divide logits by 2
    "A_generic": Detector("umm-maybe/AI-image-detector", "artificial"),
}
DET = "A_openfake"                 # the detector the A and C systems use

for name, d in DETECTORS.items():
    for c, p in zip(CASES, d([c["published"] for c in CASES])):
        c.setdefault("det", {})[name] = p

print(f"{'case':46} {'label':5} {'A_openfake':>10} {'A_generic':>10}")
for c in CASES:
    cells_ = [f"{c['det'][n]:.2f} {'✓' if (c['det'][n] >= THRESHOLD) == (c['label'] == 'fake') else '✗'}" for n in DETECTORS]
    print(f"{c['name']:46} {c['label']:5} {cells_[0]:>10} {cells_[1]:>10}")

✓ means the detector agreed with the dataset's `real`/`fake` label at the 0.5 threshold. A detector only answers condition 1: it gives the two copies of the same fake the same high score, disclosed or not.

> **⚠️ Production reality —** the OpenFake detector's model card reports that on social media it flags many **real** text graphics (screenshots, news chyrons, memes) as AI: 22 of 35 in their test. If your platform is full of screenshots, that's your false-flag rate, and no number on OpenFake will show it.

### B — a VLM

One call per image, asking for every condition the VLM can see. It cannot read file metadata, which is realistic: a VLM sees pixels. The `evidence` sentence is for a human reviewer and is **never scored**: a plausible reason is not a correct one. Answers are cached by the image's hash, so a rerun costs nothing.

In [ ]:
import groq

VLM_PROMPT = (
    "You audit images under EU AI Act Article 50(4). Look at the image and reply with ONLY a JSON object:\n"
    '{"p_fake": <0-1, probability the image is AI-generated or AI-manipulated; ordinary compression, '
    "resizing or filters are NOT manipulation>,\n"
    ' "real_entity": <true if it depicts an existing, identifiable person, place, object or event '
    "and could falsely appear authentic>,\n"
    ' "evident_fiction": <true if it is evidently artistic, satirical, fantastical or fictional>,\n'
    ' "visible_ai_label": <true if the image itself shows a visible notice that it is AI-generated>,\n'
    ' "evidence": "<one short sentence>"}'
)

def parse_json(text):
    m = re.search(r"\{.*\}", text or "", re.DOTALL)
    try:
        return json.loads(m.group()) if m else None
    except json.JSONDecodeError:
        return None

def vlm_call(raw):
    buf = io.BytesIO()
    view(raw).save(buf, format="PNG")
    resp = client.chat.completions.create(
        model=VISION_MODEL, max_tokens=1024, temperature=0,   # headroom for models that reason first
        messages=[{"role": "user", "content": [
            {"type": "text", "text": VLM_PROMPT},
            {"type": "image_url", "image_url": {"url": "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()}}]}])
    out = parse_json(resp.choices[0].message.content) or {}
    try:
        p = min(max(float(out["p_fake"]), 0.0), 1.0)
    except (KeyError, TypeError, ValueError):
        p = None                                  # unparseable: shown as such, not guessed
    return {"p_fake": p,
            **{k: out.get(k) is True for k in ("real_entity", "evident_fiction", "visible_ai_label")},
            "evidence": str(out.get("evidence", ""))[:200]}

def vlm_judge(raw):
    return cached(f"vlm|{VISION_MODEL}|{hashlib.sha256(raw).hexdigest()[:16]}", lambda: vlm_call(raw))

for c in CASES:
    try:
        c["vlm"] = vlm_judge(c["published"])
    except groq.RateLimitError as e:          # the client already retried; usually the daily cap
        print(f"⏸ VLM rate limit: {str(e)[:150]}\n  Rerun this cell later; finished calls are cached.")
        break
    v = c["vlm"]
    p = "n/a" if v["p_fake"] is None else f"{v['p_fake']:.2f}"
    print(f"{c['name']:46} p_fake {p:>4}  real_entity {v['real_entity']!s:5}  fiction {v['evident_fiction']!s:5}  "
          f"visible_label {v['visible_ai_label']!s:5}\n{'':46} {v['evidence']!r}")

Check the VLM against our hand labels: did it name the real entities, spot the anime as fiction, read the "AI-generated" box? Those are the reading tasks it's here for. Its `p_fake` is a token it wrote, not a calibrated probability: expect round numbers and confident evidence on genuinely real photos.

## 5. Compare all four systems

Now combine the tool outputs into the four systems from step 3 and score each against gold. ✓ means the system agreed with gold, ✗ that it didn't.

In [ ]:
SYSTEMS = ["A only", "A + D", "B end to end", "C hybrid"]

def flags(det, vlm, prov):
    is_fake = det[DET] >= THRESHOLD
    meaning = vlm["real_entity"] and not vlm["evident_fiction"] and not vlm["visible_ai_label"]
    return {"A only": is_fake,
            "A + D": is_fake and not prov["ai_metadata"],
            "B end to end": (vlm["p_fake"] or 0) >= 0.5 and meaning,
            "C hybrid": is_fake and meaning and not prov["ai_metadata"]}

done = [c for c in CASES if "vlm" in c]       # all nine, unless the VLM hit its limit
print(f"{'case':46} {'gold':>9}  " + " ".join(f"{s:>12}" for s in SYSTEMS))
for c in done:
    c["flags"] = flags(c["det"], c["vlm"], c["prov"])
    marks = " ".join(f"{('FLAG' if c['flags'][s] else '-') + (' ✓' if c['flags'][s] == c['gold'] else ' ✗'):>12}" for s in SYSTEMS)
    print(f"{c['name']:46} {'VIOLATION' if c['gold'] else '-':>9}  {marks}")
print()
for s in SYSTEMS:
    print(f"{s:12}: {sum(c['flags'][s] == c['gold'] for c in done)}/{len(done)} cases right")

Read it column by column:

- **A only** flags every fake, so it also "catches" the disclosed copies, the anonymous man and the anime image: every one a false flag a reviewer has to clear.
- **A + D** clears the metadata-marked copy but not the visibly labelled one: D reads files, not pixels.
- **B and C** can clear the visible label and the fiction, if the VLM reads them. Where B and C disagree, it's the "is it fake" call: the VLM's against the detector's.
- The **anonymous man in a "Bernie for President" shirt** is a judgement call on purpose: a real politician is *named*, but no real person is *depicted*. We labelled it not a deepfake; if your reviewers would disagree, that's a policy decision to write down before you measure anything.

Nine cases are a demonstration, not a measurement. To choose a design you'd score hundreds of cases per condition (exercise 1).

## 6. Try your own image

Give any image URL, or a local path (e.g. after uploading through Colab's file panel). The cell runs the three tools, then gives the **verdict** of the recommended system, C hybrid, with the reason, and shows where the other systems disagree.

Try a photo from your phone, an image downloaded straight from ChatGPT or Gemini (their files carry C2PA and IPTC marks, which D should find), and then a screenshot of that same image (the marks are gone).

> **⚠️ Privacy —** the VLM step sends the image to Groq's API. Don't use photos of private people or anything you wouldn't upload.

In [ ]:
IMAGE = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a8/Tour_Eiffel_Wikimedia_Commons.jpg/960px-Tour_Eiffel_Wikimedia_Commons.jpg"

def verdict(det, vlm, prov):
    """C hybrid's decision, in words: the first condition that clears the image, or a flag."""
    p = det[DET]
    if p < THRESHOLD:
        return False, f"the detector says it's a real photo (p_fake {p:.2f})"
    if prov["ai_metadata"]:
        return False, "AI-generated, but the file discloses it in its metadata"
    if vlm["visible_ai_label"]:
        return False, "AI-generated, but it carries a visible AI label"
    if vlm["evident_fiction"]:
        return False, "AI-generated, but evidently fiction or art"
    if not vlm["real_entity"]:
        return False, "AI-generated, but it doesn't depict a real, identifiable person or event"
    return True, "AI-generated, depicts a real entity, and isn't disclosed"

raw = http_get(IMAGE) if IMAGE.startswith("http") else open(IMAGE, "rb").read()
display(view(raw, 320))
det = {name: d([raw])[0] for name, d in DETECTORS.items()}
prov = provenance(raw)
vlm = vlm_judge(raw)

print("Tools")
for name, p in det.items():
    print(f"  {name:12}: p_fake {p:.2f} → {'AI' if p >= THRESHOLD else 'real'}")
print(f"  VLM         : p_fake {vlm['p_fake']}, real entity {vlm['real_entity']}, fiction {vlm['evident_fiction']}, "
      f"visible AI label {vlm['visible_ai_label']}\n                {vlm['evidence']!r}")
print(f"  provenance  : AI metadata {prov['ai_metadata']}, C2PA manifest {prov['c2pa_manifest']}")

flag, why = verdict(det, vlm, prov)
print(f"\nVERDICT: {'⚠️ FLAG for human review' if flag else '✅ no flag'}, because {why}.")
if (det[DET] >= THRESHOLD) != ((vlm["p_fake"] or 0) >= 0.5):
    print(f"  Note: the detector and the VLM disagree on whether it's AI-generated.")
f = flags(det, vlm, prov)
others = [s for s in SYSTEMS if f[s] != flag]
print("  Other systems: " + (", ".join(f"{s} would {'flag' if f[s] else 'not flag'} it" for s in others) if others else "all agree."))

## What this should tell you

Check these against your own output rather than taking them on faith:

- **Detection is a detector's job, but only for generators it has seen.** The OpenFake detector looks excellent here because it was trained on this dataset; the popular generic one shows what happens on generators newer than its training data. Your production detector is always somewhere between the two, and drifting toward the second.
- **Compliance is not detection.** "A only" flags every fake, disclosed or not, real entity or anime. Only systems that check provenance *and* meaning can clear those.
- **Metadata is fragile.** Any re-encode strips it (try the screenshot in step 6). Visible labels and robust watermarks survive; metadata alone is not a disclosure strategy.
- **The VLM earns its place on meaning.** Real entity, satire, a visible label: these are reading tasks, where a pixel classifier has nothing to say.

> **⭐ Key takeaway —** decompose the legal question into the conditions it's made of, give each condition the cheapest tool that answers it well, and measure each piece and the whole. Here that's a detector for *is it fake*, provenance for *was it disclosed*, a VLM for *what does it depict*, and a human for *is this a violation*. A single model asked the whole question is easier to build and harder to trust.

> **🔵 Interview signal —** the hybrid also has a cost lever B lacks: run the cheap detector on everything and call the VLM **only** on images it flags. If the detector flags 5% of traffic, VLM spend drops ~20×. Say that number in the design review.

## Exercises

1. **From demo to measurement.** Build a golden set of 50+ cases per condition: real photos from two different sources, fakes from generators newer than the detector's training data, real-entity fakes (search OpenFake rows where `prompt_kind == "published_prompt"`: user-written prompts keep the names that auto-captions drop). Report precision and recall per system ([04/01](../04-evals/01-golden-sets.ipynb)). Does the ranking from nine cases survive?
2. **Calibrate the threshold.** 0.5 is the detectors' default, not a decision. Score 100 real photos from *your* platform's kind of source and set each detector's threshold so at most 5% of them are flagged. How far does it move from 0.5?
3. **Your own photo set.** Put 20 of your own images in a folder with a CSV (`file,label,real_entity,fiction`) and run them through the step 6 code. Include phone photos, screenshots and AI images downloaded straight from a generator. Which system makes the most mistakes, and on what?
4. **Train your own detector.** Freeze a CLIP image encoder, embed 200 OpenFake training images, and fit a logistic regression on the embeddings (a linear probe, the UnivFD recipe). How close does it get to `A_openfake` on your golden set, and on generators neither has seen?
5. **Real C2PA.** Install the `c2pa-python` package and replace the byte check in `provenance` with actual manifest reading and signature validation. Test it on a C2PA-signed image straight from a generator. What does "manifest present but signature invalid" mean for the compliance decision?
6. **Hybrid economics.** Change step 6 so the VLM is called only when the detector flags. At 5M uploads/month with 1% undisclosed deepfakes, what's the VLM spend for B versus the gated hybrid, and the reviewer load (flags × 1 minute)? Write the one-sentence design-doc recommendation.